# CNN-only: tuning, huấn luyện lại và kiểm tra

Notebook này dùng cùng train/validation `obfu_http`, tokenizer ký tự, `max_len=1024`, seed 42 và ngưỡng 0,5 như phép so sánh với CNN-LSTM trên WebApp. Tám cấu hình CNN-only được chọn bằng F1 lớp tấn công trên validation; sau đó mô hình được huấn luyện lại từ đầu và kiểm tra trên test nội bộ cùng hai dataset mới đã gộp. Dataset mới chỉ dùng để test.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE if (HERE / 'cnn_only' / 'tune_cnn_only.py').is_file() else HERE.parent
if not (PROJECT_ROOT / 'cnn_only' / 'tune_cnn_only.py').is_file():
    raise FileNotFoundError('Hãy chạy notebook từ repo root hoặc thư mục cnn_only.')
TUNING_ROOT = PROJECT_ROOT / 'cnn_only' / 'artifacts_cnn_only_tuning' / 'obfu_http'
FINAL_DIR = TUNING_ROOT / 'final'
REPORT_ROOT = PROJECT_ROOT / 'reports' / 'merged_external_evaluation'
TUNED_REPORT = REPORT_ROOT / 'cnn_only_tuned'
COMPARISON_PATH = REPORT_ROOT / 'comparison_tuned_1024.csv'
RUN_TRAINING = not (FINAL_DIR / 'metadata_and_results.json').is_file()
report_summary = TUNED_REPORT / 'summary.json'
final_checkpoint = FINAL_DIR / 'best_cnn_only.keras'
RUN_EXTERNAL_TEST = (not report_summary.is_file()) or (
    final_checkpoint.is_file() and report_summary.stat().st_mtime < final_checkpoint.stat().st_mtime
)
# Set either flag to True to force the corresponding step to run again.
print('Python:', sys.executable)
print('Project:', PROJECT_ROOT)


Python: C:\Users\admin\Desktop\obfuscated-web-attack-detection\.venv-webapp\Scripts\python.exe
Project: C:\Users\admin\Desktop\obfuscated-web-attack-detection


## 1. Tuning và train lại

Tìm kiếm 8 cấu hình riêng biệt, tối đa 12 epoch/trial và dừng sớm sau 2 epoch. Chọn bằng F1 validation tại ngưỡng 0,5, dùng validation loss để phá hòa. Train lại cấu hình tốt nhất tối đa 50 epoch và dừng sớm sau 3 epoch, giống bước cuối của CNN-LSTM. Có thể chạy lại notebook để tiếp tục các trial đã hoàn thành.


In [2]:
if RUN_TRAINING:
    script = PROJECT_ROOT / 'cnn_only' / 'tune_cnn_only.py'
    env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL='2')
    for trial_number in range(1, 9):
        print(f'CNN-only tuning trial {trial_number}/8', flush=True)
        subprocess.run([sys.executable, '-u', str(script), '--only-trial', str(trial_number)],
                       cwd=PROJECT_ROOT, env=env, check=True)
    subprocess.run([sys.executable, '-u', str(script), '--final-only'],
                   cwd=PROJECT_ROOT, env=env, check=True)
else:
    print('Dùng artifact đã train tại:', FINAL_DIR)


Dùng artifact đã train tại: C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_only\artifacts_cnn_only_tuning\obfu_http\final


In [3]:
protocol = json.loads((TUNING_ROOT / 'search_protocol.json').read_text(encoding='utf-8'))
trials = pd.read_csv(TUNING_ROOT / 'tuning_results.csv').sort_values(
    ['val_attack_f1_at_0_5', 'min_val_loss'], ascending=[False, True]
)
selected = json.loads((TUNING_ROOT / 'best_hyperparameters.json').read_text(encoding='utf-8'))
baseline_splits = PROJECT_ROOT / 'cnn_only' / 'artifacts_cnn_only_matched_1024' / 'processed_data_by_dataset' / 'obfu_http'
if baseline_splits.is_dir():
    for split_name, expected_hash in protocol['split_sha256'].items():
        other = baseline_splits / f'{split_name}.csv'
        assert other.is_file() and __import__('hashlib').sha256(other.read_bytes()).hexdigest() == expected_hash
    print('Train/val/test splits are byte-identical to the prior CNN-only 1024 run.')
print('Train/val source:', protocol['source'], '| max_len:', protocol['max_len'], '| threshold:', protocol['threshold'])
print('Cấu hình được chọn:', selected)
display(trials.drop(columns=['checkpoint']).reset_index(drop=True).style.format({
    'min_val_loss': '{:.6f}', 'val_attack_f1_at_0_5': '{:.6f}'
}))


Train/val/test splits are byte-identical to the prior CNN-only 1024 run.
Train/val source: obfu_http | max_len: 1024 | threshold: 0.5
Cấu hình được chọn: {'embedding_dim': 64, 'cnn_filters': 64, 'dense_units': 64, 'dropout': 0.4, 'learning_rate': 0.0003, 'batch_size': 128}


,trial,embedding_dim,cnn_filters,dense_units,dropout,learning_rate,batch_size,epochs_ran,min_val_loss,val_attack_f1_at_0_5,parameter_count
0,3,64,64,64,0.400000,0.000300,128,12,0.000674,0.999780,44289
1,6,32,64,32,0.200000,0.000300,128,12,0.001196,0.999634,32449
2,1,64,128,64,0.300000,0.001000,128,5,0.001527,0.999560,122241
3,7,32,128,64,0.400000,0.001000,128,5,0.001483,0.999487,106369
4,2,32,64,64,0.200000,0.000300,256,12,0.001846,0.999194,34561
5,8,64,128,32,0.400000,0.001000,256,8,0.002392,0.999120,118081
6,5,32,128,32,0.200000,0.000300,256,12,0.001827,0.999047,102209
7,4,64,128,32,0.400000,0.001000,128,5,0.004084,0.998753,118081


## 2. Kết quả test nội bộ

Test nội bộ của `obfu_http` chỉ được đọc sau khi đã lưu cấu hình tốt nhất.


In [4]:
final = json.loads((FINAL_DIR / 'metadata_and_results.json').read_text(encoding='utf-8'))
internal = final['evaluation']['test']
print('Selected trial:', final['selected_trial'], '| final epochs:', final['model']['epochs_run'],
      '| parameters:', final['model']['parameter_count'])
display(pd.DataFrame([{
    'Dataset': 'obfu_http internal test',
    'Threshold': 0.5,
    'Accuracy': internal['accuracy'],
    'Attack Precision': internal['classification_report']['Attack (1)']['precision'],
    'Attack Recall': internal['classification_report']['Attack (1)']['recall'],
    'Attack F1': internal['classification_report']['Attack (1)']['f1-score'],
}]).style.format({key: '{:.4f}' for key in ['Accuracy', 'Attack Precision', 'Attack Recall', 'Attack F1']}))


Selected trial: 3 | final epochs: 18 | parameters: 44289


,Dataset,Threshold,Accuracy,Attack Precision,Attack Recall,Attack F1
0,obfu_http internal test,0.500000,0.9994,0.9999,0.9990,0.9994


## 3. Test ngoài trên hai dataset mới đã gộp

Tiền xử lý giống WebApp và notebook CNN-LSTM; mỗi bộ chỉ bỏ các hàng gốc trùng hoàn toàn trong chính bộ đó. Ngưỡng cố định 0,5.


In [5]:
if RUN_EXTERNAL_TEST:
    subprocess.run([
        sys.executable, '-u', str(PROJECT_ROOT / 'analysis' / 'evaluate_merged_cnn_only.py'),
        '--model-dir', str(FINAL_DIR), '--output-dir', str(TUNED_REPORT), '--threshold', '0.5',
    ], cwd=PROJECT_ROOT, env=dict(os.environ, TF_CPP_MIN_LOG_LEVEL='2'), check=True)
    subprocess.run([
        sys.executable, str(PROJECT_ROOT / 'analysis' / 'compare_merged_models.py'),
        '--cnn-only-summary', str(TUNED_REPORT / 'summary.json'),
        '--cnn-only-label', 'CNN-only tuned (1024)',
        '--output', str(COMPARISON_PATH),
    ], cwd=PROJECT_ROOT, check=True)
else:
    print('Dùng kết quả test ngoài đã lưu tại:', TUNED_REPORT)


Dùng kết quả test ngoài đã lưu tại: C:\Users\admin\Desktop\obfuscated-web-attack-detection\reports\merged_external_evaluation\cnn_only_tuned


## 4. Bảng hiệu suất cuối cùng

Hai mô hình được so trên cùng các hàng đã tiền xử lý, `max_len=1024` và threshold 0,5. Cột FP/FN là số lượng lỗi, Attack F1 là F1 của lớp tấn công.


In [6]:
comparison = pd.read_csv(COMPARISON_PATH)
assert set(comparison['Threshold']) == {0.5}
assert set(comparison['Max_len']) == {1024}
columns = ['Model', 'Dataset', 'Rows', 'Threshold', 'Accuracy', 'AUC-ROC',
           'PR-AUC', 'Attack Precision', 'Attack Recall', 'Attack F1', 'FP', 'FN']
display(comparison[columns].style.format({key: '{:.4f}' for key in [
    'Accuracy', 'AUC-ROC', 'PR-AUC', 'Attack Precision', 'Attack Recall', 'Attack F1'
]}))


,Model,Dataset,Rows,Threshold,Accuracy,AUC-ROC,PR-AUC,Attack Precision,Attack Recall,Attack F1,FP,FN
0,CNN-LSTM (WebApp),merged_all,156186,0.500000,0.8095,0.9033,0.8976,0.8731,0.7459,0.8045,8898,20854
1,CNN-LSTM (WebApp),obfuscated_grouped,134777,0.500000,0.7901,0.8885,0.8446,0.8434,0.6870,0.7572,8192,20098
2,CNN-LSTM (WebApp),xss_payloads_with_obfuscated,21409,0.500000,0.9317,0.9658,0.9937,0.9604,0.9577,0.9591,706,756
3,CNN-only tuned (1024),merged_all,156186,0.500000,0.8311,0.9059,0.9025,0.8654,0.8037,0.8334,10264,16114
4,CNN-only tuned (1024),obfuscated_grouped,134777,0.500000,0.8140,0.8923,0.8656,0.8349,0.7598,0.7956,9650,15419
5,CNN-only tuned (1024),xss_payloads_with_obfuscated,21409,0.500000,0.9389,0.9605,0.9914,0.9655,0.9611,0.9633,614,695
